# Algorithmic Resource & Telemetry Analytics

**Student:** Sahil Rashid Kachroo  
**Programme:** AICTE | IBM SkillsBuild Data Analytics with AI Internship  

---

## Project Overview

This notebook presents an end-to-end data analytics pipeline for **Algorithmic Resource & Telemetry Analytics**. The project simulates the kind of low-level performance telemetry collected from C/C++ runtimes and cloud workloads, then applies the full data science workflow:

1. **Synthetic data generation** — realistic simulation of execution latency and heap-memory footprints for three canonical algorithm families across a wide range of input sizes.
2. **Data cleaning & feature engineering** — rolling-median smoothing, IQR-based outlier clipping, and log-scale transformation.
3. **Exploratory Data Analysis (EDA)** — log-log latency plots, memory growth curves, and a feature correlation matrix.
4. **Machine Learning pipeline** — Linear Regression, Polynomial Regression (degree 3), and Random Forest Regressor evaluated on R², MAE, and RMSE.
5. **Cloud cost optimisation analysis** — normalised cost modelling and risk-threshold identification per algorithm to support infrastructure sizing decisions.

> **Objective:** Demonstrate how data-driven telemetry analytics can surface hidden complexity costs, guide algorithm selection, and drive cloud spending optimisation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Environment ready.')

SECTION 2: Data Grounding - C/C++ Hardware Telemetry Ingestion
Ingesting real hardware telemetry data collected from the C++ benchmarking script (`telemetry_raw.csv`).

In [ ]:
import pandas as pd
import os

print('Loading C/C++ Hardware Telemetry...')
df_raw = pd.read_csv('telemetry_raw.csv', dtype={'algorithm':'category', 'input_size':'int64', 'execution_time_us':'float64', 'memory_bytes':'float64'})
print(df_raw.shape)
df_raw.head(9)


---

## Section 3 — Data Cleaning & Preprocessing

Raw telemetry suffers from three common issues addressed here:

### 3.1 Rolling Median Smoothing
A centred window of size **W = 5** suppresses momentary spikes caused by OS interrupts or garbage-collector pauses without distorting long-term trends.

### 3.2 IQR-Based Outlier Clipping
Per-group Inter-Quartile Range clipping (1.5 × IQR fence) removes extreme tail observations that would otherwise inflate RMSE during model training.

### 3.3 Log-Scale Feature Engineering
Because both `input_size` and the target metrics span several orders of magnitude, `log1p`-transformed features (`log_input_size`, `log_exec_time`, `log_memory`) linearise the relationships and improve the conditioning of regression models.

A `LabelEncoder` converts the categorical `algorithm` column to an ordinal integer `algo_encoded` for use in scikit-learn estimators.

In [ ]:
WINDOW = 5
def rolling_median_smooth(s):
    return s.rolling(window=WINDOW, min_periods=1, center=True).median()
df = df_raw.copy()
df['execution_time_us'] = (df.groupby('algorithm', observed=True)['execution_time_us']
                             .transform(rolling_median_smooth))
df['memory_bytes'] = (df.groupby('algorithm', observed=True)['memory_bytes']
                        .transform(rolling_median_smooth))
# IQR outlier clipping per group
for col in ['execution_time_us', 'memory_bytes']:
    Q1 = df.groupby('algorithm', observed=True)[col].transform('quantile', 0.25)
    Q3 = df.groupby('algorithm', observed=True)[col].transform('quantile', 0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lo, upper=hi)
df['log_input_size'] = np.log1p(df['input_size'])
df['log_exec_time']  = np.log1p(df['execution_time_us'])
df['log_memory']     = np.log1p(df['memory_bytes'])
le = LabelEncoder()
df['algo_encoded'] = le.fit_transform(df['algorithm'])
print('Cleaned shape:', df.shape)
df.describe()
# Save cleaned telemetry dataset
df.to_csv('telemetry_clean.csv', index=False)
print('Saved telemetry_clean.csv')


---

## Section 4 — EDA: Exploratory Data Analysis

Three complementary visualisations reveal the structure of the telemetry:

### 4.1 Execution Latency (log-log plot)
Plotting on a log-log scale converts power-law relationships into straight lines, making complexity class immediately visible. `binary_exponentiation` shows a gentle super-logarithmic slope; both linear algorithms produce parallel lines with slopes near 1.

### 4.2 Memory Growth Curves
The filled area chart highlights the gap between O(1) memory (`binary_exponentiation`) and the O(N) algorithms. `linked_list_traversal` diverges sharply at large N due to per-node pointer overhead (16 B/node vs. ~8 B amortised for vector doubling).

### 4.3 Feature Correlation Matrix
The heatmap quantifies linear relationships among engineered features. `log_input_size` is strongly correlated with both resource targets, validating it as the primary predictor.

In [ ]:
PALETTE = {'binary_exponentiation': '#00C8FF',
           'vector_reallocation':   '#FF6B6B',
           'linked_list_traversal': '#A8FF78'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.suptitle('EDA — Algorithmic Resource Telemetry', fontsize=14, fontweight='bold')

# Plot 1: Execution time (log-log)
ax = axes[0]
for algo, grp in df.groupby('algorithm', observed=True):
    ax.plot(grp['input_size'], grp['execution_time_us'],
            label=algo, color=PALETTE[algo], linewidth=2)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Execution Latency (log-log)'); ax.set_xlabel('N'); ax.set_ylabel('\u00b5s')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Plot 2: Memory growth
ax = axes[1]
for algo, grp in df.groupby('algorithm', observed=True):
    ax.fill_between(grp['input_size'], grp['memory_bytes']/1024, alpha=0.3, color=PALETTE[algo])
    ax.plot(grp['input_size'], grp['memory_bytes']/1024, label=algo, color=PALETTE[algo], linewidth=2)
ax.set_xscale('log')
ax.set_title('Memory Growth (KB)'); ax.set_xlabel('N'); ax.set_ylabel('KB')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Plot 3: Correlation matrix
ax = axes[2]
corr = df[['log_input_size','execution_time_us','memory_bytes','algo_encoded']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA complete.')

---

## Section 5 — Machine Learning Pipeline

We train three regression models on two targets (`execution_time_us` and `memory_bytes`) using a common 80/20 train-test split.

| Model | Rationale |
|---|---|
| **Linear Regression** | Baseline; works well when log-features linearise the relationship |
| **Polynomial Regression (degree 3)** | Captures non-linear residuals via feature expansion |
| **Random Forest (300 trees)** | Non-parametric; robust to interactions and heteroscedasticity |

**Evaluation metrics**
- **MAE** (Mean Absolute Error) — interpretable average error in original units  
- **RMSE** (Root Mean Squared Error) — penalises large errors more heavily  
- **R²** (Coefficient of Determination) — proportion of variance explained  

The best-performing Random Forest models are serialised to disk via `joblib` for downstream deployment or API serving.

In [ ]:
FEATURES = ['log_input_size', 'algo_encoded']
TARGET_TIME = 'execution_time_us'
TARGET_MEM  = 'memory_bytes'

X = df[FEATURES].values
y_time = df[TARGET_TIME].values
y_mem  = df[TARGET_MEM].values

X_tr, X_te, yt_tr, yt_te = train_test_split(X, y_time, test_size=0.2, random_state=42)
_, __, ym_tr, ym_te       = train_test_split(X, y_mem,  test_size=0.2, random_state=42)

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = mean_squared_error(y_te, pred) ** 0.5
    r2   = r2_score(y_te, pred)
    print(f'{name:35s}  MAE={mae:>12.3f}  RMSE={rmse:>12.3f}  R\u00b2={r2:.6f}')
    return model, pred

print(f'{"Model":<35}  {"MAE":>14}  {"RMSE":>14}  R\u00b2')
print('-'*80)

lr   = LinearRegression()
poly_pipe = Pipeline([('poly', PolynomialFeatures(degree=3, include_bias=False)),
                      ('lr',   LinearRegression())])
rf   = RandomForestRegressor(n_estimators=300, max_features='sqrt',
                              min_samples_leaf=2, n_jobs=-1, random_state=42)

print('--- Execution Time ---')
lr_t,   pred_lr_t   = evaluate('Linear Regression',        lr,        X_tr, X_te, yt_tr, yt_te)
poly_t, pred_poly_t = evaluate('Polynomial Regression(3)', poly_pipe, X_tr, X_te, yt_tr, yt_te)
rf_t,   pred_rf_t   = evaluate('Random Forest',             rf,        X_tr, X_te, yt_tr, yt_te)

print()
print('--- Memory Bytes ---')
lr_m,   pred_lr_m   = evaluate('Linear Regression',        LinearRegression(), X_tr, X_te, ym_tr, ym_te)
poly_m, pred_poly_m = evaluate('Polynomial Regression(3)', Pipeline([('poly', PolynomialFeatures(degree=3, include_bias=False)), ('lr', LinearRegression())]), X_tr, X_te, ym_tr, ym_te)
rf_m,   pred_rf_m   = evaluate('Random Forest',             RandomForestRegressor(n_estimators=300, max_features='sqrt', min_samples_leaf=2, n_jobs=-1, random_state=42), X_tr, X_te, ym_tr, ym_te)

In [ ]:
joblib.dump(rf_t, 'model_time.pkl', compress=3)
joblib.dump(rf_m, 'model_mem.pkl',  compress=3)
print('Models saved.')

# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.suptitle('ML Model \u2014 Actual vs Predicted', fontsize=13, fontweight='bold')

for ax, y_te, y_pred, title in [
    (axes[0], yt_te, pred_rf_t, 'Execution Time (\u00b5s)'),
    (axes[1], ym_te, pred_rf_m, 'Memory (bytes)')]:
    ax.scatter(y_te, y_pred, alpha=0.6, s=25, color='#00C8FF')
    lim = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ml_actual_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()

---

## Section 6 — Business & Cloud Cost Optimisation Analysis

### 6.1 Cloud Cost Modelling
Modern serverless and microservice platforms bill at sub-millisecond granularity. We normalise execution time to a **cost unit** of $1 \times 10^{-9}$ USD per microsecond, representative of AWS Lambda / GCP Cloud Run tiered pricing at scale.

### 6.2 Risk Threshold Identification
A **10× baseline cost multiple** is used as a practical risk trigger — the point at which an engineering team should re-evaluate the algorithm choice, introduce caching, or provision dedicated compute. The shaded regions and vertical markers in the plot identify these critical N values for each algorithm.

### 6.3 Business Recommendations

| Algorithm | Risk Threshold | Recommendation |
|---|---|---|
| `binary_exponentiation` | Very high N | Safe for all practical input sizes; no action needed |
| `vector_reallocation` | Medium N | Pre-allocate capacity with `reserve()` to eliminate realloc churn |
| `linked_list_traversal` | Low N | Replace with contiguous data structures (e.g., `std::deque` or array-backed list) to exploit CPU cache locality |

> **Key insight:** Memory allocation patterns, not just time complexity, drive cloud costs at scale. A linked list with identical O(N) complexity to a vector incurs up to **2× the cost** due to allocator overhead and cache misses.

In [ ]:
COST_PER_US   = 1e-9  # $ per microsecond (normalised cloud unit)
RISK_MULTIPLE = 10

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Cloud Cost Risk Threshold by Algorithm', fontsize=13, fontweight='bold')

for algo, grp in df.groupby('algorithm', observed=True):
    grp = grp.sort_values('input_size')
    cost = grp['execution_time_us'] * COST_PER_US
    baseline = cost.iloc[0] if cost.iloc[0] > 0 else cost[cost > 0].iloc[0]
    risk_mask = cost >= baseline * RISK_MULTIPLE
    risk_n = grp['input_size'][risk_mask].min() if risk_mask.any() else None
    c = PALETTE[algo]
    ax.plot(grp['input_size'], cost, color=c, linewidth=2.2, label=algo)
    if risk_n:
        ax.fill_between(grp['input_size'], cost,
                        where=(grp['input_size'] >= risk_n), alpha=0.2, color=c)
        ax.axvline(risk_n, color=c, linestyle=':', linewidth=1.2)
        ax.text(risk_n, cost.max()*0.85, f' Risk\n>{risk_n:,}', color=c, fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Input Size N'); ax.set_ylabel('Normalised Cost ($)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cost_threshold.png', dpi=120, bbox_inches='tight')
plt.show()

print('=== Business Summary ===')
for algo, grp in df.groupby('algorithm', observed=True):
    grp = grp.sort_values('input_size')
    cost = grp['execution_time_us'] * COST_PER_US
    baseline = cost.iloc[0] if cost.iloc[0] > 0 else cost[cost > 0].iloc[0]
    risk_mask = cost >= baseline * RISK_MULTIPLE
    risk_n = grp['input_size'][risk_mask].min() if risk_mask.any() else 'Never'
    print(f'{algo}: cost risk threshold at N={risk_n}')